# Week 46, Day 5: Serialising trees

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w046d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def preorder(n):
    return [] if n is None else [n.val] + preorder(n.left) + preorder(n.right)

T = TreeNode
a = T(2, T(1))                 # 1 is the left child of 2
b = T(2, None, T(1))           # 1 is the right child of 2: not a BST
print(preorder(a), preorder(b), preorder(a) == preorder(b))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def load_bst(values):
    position = [0]
    def build(bound):
        i = position[0]
        if i == len(values) or values[i] > bound:
            return None
        node = TreeNode(values[i])
        position[0] += 1
        node.left = build(node.val)       # left subtree: values below this one
        node.right = build(bound)         # right subtree: values up to the old bound
        return node
    return build(float("inf"))

def preorder(n):
    return [] if n is None else [n.val] + preorder(n.left) + preorder(n.right)

def inorder(n):
    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)

saved = "50 30 20 40 35 70 60 80 90"
root = load_bst([int(v) for v in saved.split()])
print(preorder(root) == [int(v) for v in saved.split()], inorder(root))

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def to_list(root):
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def from_list(items):
    if not items or items[0] is None:
        return None
    root = TreeNode(items[0])
    queue, i = deque([root]), 1
    while queue and i < len(items):
        parent = queue.popleft()
        for side in ("left", "right"):
            if i < len(items) and items[i] is not None:
                child = TreeNode(items[i])
                setattr(parent, side, child)
                queue.append(child)
            i += 1
    return root

T = TreeNode
saved = to_list(T(1, T(2, None, T(4)), T(3, None, T(5))))
print(saved, to_list(from_list(saved)) == saved)

In [ ]:
import json

class Node:
    def __init__(self, name, children=None):
        self.name, self.children = name, children or []

def to_dict(node):
    return {"name": node.name, "children": [to_dict(c) for c in node.children]}

def from_dict(d):
    return Node(d["name"], [from_dict(c) for c in d["children"]])

menu = Node("File", [Node("New"), Node("Open", [Node("Recent"), Node("Browse")]), Node("Quit")])
text = json.dumps(to_dict(menu))
print(text)
again = from_dict(json.loads(text))
print([c.name for c in again.children], again.children[1].children[0].name)

In [ ]:
import json

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

class Codec:
    VERSION = 2

    def dumps(self, root):
        return json.dumps({"format": "tree", "version": self.VERSION, "root": self._enc(root)})

    def _enc(self, n):
        return None if n is None else [n.val, self._enc(n.left), self._enc(n.right)]

    def loads(self, text):
        data = json.loads(text)
        if data.get("version") != self.VERSION:
            raise ValueError(f"unsupported version: {data.get('version')!r}")
        return self._dec(data["root"])

    def _dec(self, item):
        return None if item is None else TreeNode(item[0], self._dec(item[1]), self._dec(item[2]))

c = Codec()
text = c.dumps(TreeNode("b", TreeNode("a"), TreeNode("c d")))
print(text)
print(c.dumps(c.loads(text)) == text)
try:
    c.loads('{"format": "tree", "version": 3, "root": null}')
except ValueError as e:
    print("ValueError:", e)

In [ ]:
def check_node(item, depth=0, limit=100):
    if depth > limit:
        raise ValueError("tree too deep")
    if item is None:
        return 0
    if not (isinstance(item, list) and len(item) == 3):
        raise ValueError(f"bad node: {item!r}")
    if not isinstance(item[0], int) or isinstance(item[0], bool):
        raise ValueError(f"bad value: {item[0]!r}")
    return 1 + check_node(item[1], depth + 1, limit) + check_node(item[2], depth + 1, limit)

for data in [[2, [1, None, None], None], [2, [1, None], None], [True, None, None], "tree"]:
    try:
        print("ok,", check_node(data), "nodes")
    except ValueError as e:
        print("ValueError:", e)

> **Key idea.** A BST can be saved as its plain pre-order values: the loader rebuilds it in O(n) by passing an upper bound down, or with a monotonic stack. Level-order lists use None for missing children of real nodes, with trailing Nones dropped; load them with a queue. General trees suit nested JSON, but deep nesting hits recursion limits, so very deep trees use a flat list of id and parent records. Put a format name and version in saved data, keep `dumps` and `loads` together in one class, and test with round trips over many random trees. Treat loaded data as untrusted: check types, structure, size and depth, raise one clear `ValueError`, and never use `eval` or `pickle` on it.

### Exercise: A BST from its values

Write `save_bst(root)` (the pre-order values separated by spaces; `""` for an empty tree) and `load_bst(text)`, rebuilding it in O(n) with a **monotonic stack** and no recursion. The checks load a 20,000-value chain.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def save_bst(root):
    ___

def load_bst(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef ins(n, v):\n    if n is None:\n        return TreeNode(v)\n    if v < n.val:\n        n.left = ins(n.left, v)\n    else:\n        n.right = ins(n.right, v)\n    return n\ndef shape(n):\n    out, stack = [], [n]\n    while stack:\n        x = stack.pop()\n        if x is None:\n            out.append(\"#\")\n        else:\n            out.append(x.val)\n            stack += [x.right, x.left]\n    return out\ndef test_roundtrip():\n    \"round trips\"\n    rng = random.Random(16)\n    for _ in range(100):\n        root = None\n        for v in rng.sample(range(-500, 500), rng.randint(0, 40)):\n            root = ins(root, v)\n        assert shape(load_bst(save_bst(root))) == shape(root)\n    assert save_bst(None) == \"\" and load_bst(\"\") is None\n    T = TreeNode\n    assert save_bst(T(50, T(30, T(20), T(40, T(35))), T(70))) == \"50 30 20 40 35 70\"\ndef test_long():\n    \"a chain, quickly\"\n    text = \" \".join(str(v) for v in range(20_000, 0, -1))\n    t0 = time.perf_counter()\n    root = load_bst(text)\n    assert time.perf_counter() - t0 < 2 and save_bst(root) == text")

### Exercise: Could this be a BST's pre-order?

Write `is_bst_preorder(values)`: whether some BST with distinct values has exactly this pre-order, in O(n) with a stack. Keep a **lower bound**: when a value pops smaller values off the stack (moving into some node's right subtree), the last popped becomes the bound, and every later value must exceed it.

In [ ]:
def is_bst_preorder(values):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef ins(n, v):\n    if n is None:\n        return [v, None, None]\n    i = 1 if v < n[0] else 2\n    n[i] = ins(n[i], v)\n    return n\ndef pre(n):\n    return [] if n is None else [n[0]] + pre(n[1]) + pre(n[2])\ndef test_check():\n    \"checking\"\n    assert is_bst_preorder([50, 30, 20, 40, 35, 70, 60, 80]) and is_bst_preorder([]) and is_bst_preorder([1])\n    assert not is_bst_preorder([5, 2, 6, 1, 7]), \"1 comes after moving right of 5\"\n    assert not is_bst_preorder([2, 2])\n    good = set()\n    for perm in itertools.permutations(range(5)):\n        root = None\n        for v in perm:\n            root = ins(root, v)\n        good.add(tuple(pre(root)))\n    for perm in itertools.permutations(range(5)):\n        assert is_bst_preorder(list(perm)) == (perm in good), perm")

### Exercise: Level-order lists

Write `to_list(root)` and `from_list(items)` for level-order lists with `None` for missing children and trailing `None`s dropped. Then `to_json(root)` and `from_json(text)`, which use the same list written as JSON (so `None` appears as `null`).

In [ ]:
import json
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def to_list(root):
    ___

def from_list(items):
    ___

def to_json(root):
    ___

def from_json(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nT = TreeNode\ndef shape(n):\n    return None if n is None else (n.val, shape(n.left), shape(n.right))\ndef rand_tree(rng, n):\n    if n == 0:\n        return None\n    k = rng.randint(0, n - 1)\n    return T(rng.randint(-9, 9), rand_tree(rng, k), rand_tree(rng, n - 1 - k))\ndef test_lists():\n    \"level-order lists\"\n    t = T(1, T(2, None, T(4)), T(3, None, T(5)))\n    assert to_list(t) == [1, 2, 3, None, 4, None, 5] and to_list(None) == [] and from_list([]) is None\n    assert shape(from_list([5, None, 7, 6])) == (5, None, (7, (6, None, None), None))\n    assert to_json(t) == \"[1, 2, 3, null, 4, null, 5]\" and shape(from_json(\"[1, null, 2]\")) == (1, None, (2, None, None))\n    rng = random.Random(17)\n    for _ in range(200):\n        t = rand_tree(rng, rng.randint(0, 30))\n        assert shape(from_list(to_list(t))) == shape(t) and shape(from_json(to_json(t))) == shape(t)")

### Exercise: Deep trees, flat records

General trees (`Node` with `name` and `children`) can be too deep for nested JSON. Write `to_records(root)`: a list of dicts `{"id": i, "parent": p, "name": ...}` in pre-order, with ids 0, 1, 2, ... and `parent` `None` for the root; and `from_records(records)`, rebuilding the tree (children in record order). Both must use loops, not recursion: the checks use a chain 50,000 deep and go through `json.dumps` and `json.loads`.

In [ ]:
class Node:
    def __init__(self, name, children=None):
        self.name, self.children = name, children or []

def to_records(root):
    ___

def from_records(records):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import json\ndef names(n):\n    out, stack = [], [n]\n    while stack:\n        x = stack.pop()\n        out.append((x.name, len(x.children)))\n        stack.extend(reversed(x.children))\n    return out\ndef test_records():\n    \"records\"\n    menu = Node(\"File\", [Node(\"New\"), Node(\"Open\", [Node(\"Recent\"), Node(\"Browse\")]), Node(\"Quit\")])\n    recs = to_records(menu)\n    assert recs[:3] == [{\"id\": 0, \"parent\": None, \"name\": \"File\"}, {\"id\": 1, \"parent\": 0, \"name\": \"New\"}, {\"id\": 2, \"parent\": 0, \"name\": \"Open\"}]\n    assert recs[3] == {\"id\": 3, \"parent\": 2, \"name\": \"Recent\"} and len(recs) == 6\n    assert names(from_records(json.loads(json.dumps(recs)))) == names(menu)\n    assert to_records(None) == [] and from_records([]) is None\ndef test_deep():\n    \"very deep\"\n    root = Node(0)\n    node = root\n    for i in range(1, 50_000):\n        child = Node(i)\n        node.children.append(child)\n        node = child\n    again = from_records(json.loads(json.dumps(to_records(root))))\n    n, depth = again, 0\n    while n.children:\n        n, depth = n.children[0], depth + 1\n    assert depth == 49_999 and n.name == 49_999")

### Exercise: A careful loader

Write `safe_load(text, max_nodes=1000)`, which loads a level-order JSON list (as in "Level-order lists") **from an untrusted source**. Return the tree, or raise `ValueError` (and only `ValueError`) if:

- the text isn't valid JSON (catch `json.JSONDecodeError`, a subclass of `ValueError`, and re-raise with your own message), or isn't a list;
- any item is neither `None` nor an `int` (and `bool` doesn't count as an `int`);
- the list is longer than `2 * max_nodes + 1` items, or holds more than `max_nodes` values;
- the first item is `None` but others follow, or the list ends with `None`;
- there are more items than the parents can take (each real node takes exactly two slots).

In [ ]:
import json
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def safe_load(text, max_nodes=1000):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def shape(n):\n    return None if n is None else (n.val, shape(n.left), shape(n.right))\ndef test_good():\n    \"good input\"\n    assert shape(safe_load(\"[1, 2, 3, null, 4]\")) == (1, (2, None, (4, None, None)), (3, None, None))\n    assert safe_load(\"[]\") is None and shape(safe_load(\"[7]\")) == (7, None, None)\ndef test_bad():\n    \"bad input\"\n    bad = [\"\", \"[1, 2\", '{\"a\": 1}', \"[1, true]\", '[1, \"2\"]', \"[1.5]\", \"[null, 1]\", \"[1, 2, null]\",\n           \"[1, null, null, 4]\", \"[1, [2]]\", \"5\", \"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]\"]\n    for text in bad:\n        try:\n            safe_load(text, max_nodes=10)\n            assert False, f\"accepted {text!r}\"\n        except ValueError:\n            pass\n    big = \"[\" + \", \".join([\"1\"] * 11) + \"]\"\n    try:\n        safe_load(big, max_nodes=10)\n        assert False, \"too many values\"\n    except ValueError:\n        pass\n    assert shape(safe_load(\"[\" + \", \".join([\"1\"] * 10) + \"]\", max_nodes=10)) is not None")

### Bug hunt: Children for nobody

This loader gives children to `None` slots as well as real nodes, by queueing missing children as if they were nodes; the later items land in the wrong places and it crashes. Only queue real children, so it prints `[1, None, 2, 3]`.

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def from_list(items):
    root = TreeNode(items[0])
    queue, i = deque([root]), 1
    while queue and i < len(items):
        parent = queue.popleft()
        for side in ("left", "right"):
            child = TreeNode(items[i]) if i < len(items) and items[i] is not None else None
            if parent is not None:
                setattr(parent, side, child)
            queue.append(child)
            i += 1
    return root

root = from_list([1, None, 2, 3])
print([root.val, root.left, root.right.val, root.right.left.val])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"only real nodes\"\n    expect_output(\"[1, None, 2, 3]\")")

### Bug hunt: The bound that never tightens

This BST loader passes the same upper bound to the left subtree as to the right, so every value after the root ends up on the left: the tree comes back as a chain and saving it again gives the wrong pre-order. The left subtree's bound should be the node's own value. Fix it so it prints `True`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def load_bst(values):
    position = [0]
    def build(bound):
        i = position[0]
        if i == len(values) or values[i] > bound:
            return None
        node = TreeNode(values[i])
        position[0] += 1
        node.left = build(bound)
        node.right = build(bound)
        return node
    return build(float("inf"))

def inorder(n):
    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)

values = [50, 30, 20, 40, 70, 60]
print(inorder(load_bst(values)) == sorted(values))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"tighten on the left\"\n    expect_output(\"True\")")

### Bug hunt: A string is not a list

This loader trusts that `children` is a list. Given a string, it loops over the characters, treats each one as a node, and reports `bad tree: each node needs a name`, which points at the wrong problem. Check the type and raise `ValueError("children must be a list")`, so it prints `bad tree: children must be a list`.

In [ ]:
import json

class Node:
    def __init__(self, name, children=None):
        self.name, self.children = name, children or []

def from_dict(d):
    if not isinstance(d, dict) or not isinstance(d.get("name"), str):
        raise ValueError("each node needs a name")
    kids = d.get("children", [])
    return Node(d["name"], [from_dict(c) for c in kids])

try:
    from_dict(json.loads('{"name": "File", "children": "ab"}'))
except ValueError as e:
    print("bad tree:", e)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"check the type\"\n    expect_output(\"bad tree: children must be a list\")\n    assert from_dict({\"name\": \"a\", \"children\": [{\"name\": \"b\"}]}).children[0].name == \"b\"")

### Puzzle: Rebuilding a BST from pre-order

Arrange the lines into the bound-passing loader. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# The tree:        8
#                /   \
#               3     10
#              / \      \
#             1   6      14

p = ___   # its pre-order values (a list), all a BST needs
q = ___   # its level-order list, with None for missing children and trailing Nones dropped
r = ___   # the in-order traversal of load_bst([5, 3, 4, 9, 7])
s = ___   # True or False: is [5, 3, 6, 4] the pre-order of some BST?
t = ___   # True or False: isinstance(True, int)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [8, 3, 1, 6, 10, 14] and q == [8, 3, 10, 1, 6, None, 14] and r == [3, 4, 5, 7, 9], \"p, q and r\"\n    assert s is False and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a compact binary format

Text is easy to read but large. Write `pack(root)` and `unpack(data)` for binary trees of integers, producing and reading `bytes`:

- 4 bytes: the number of nodes n, as an unsigned big-endian integer (`n.to_bytes(4, "big")`);
- the tree's **shape** in pre-order, one bit per position: 1 for a node, 0 for a missing child (so 2n + 1 bits), packed 8 to a byte, first bit in the highest place, the last byte padded with zero bits;
- then each value in pre-order as 4 bytes, signed big-endian (`v.to_bytes(4, "big", signed=True)`; values must fit, or raise `ValueError`).

`unpack` reverses it with `int.from_bytes(...)` and must raise `ValueError` (and nothing else) for any malformed input: too short, too long, shape bits that don't describe exactly n nodes, or non-zero padding. Use loops and stacks, not recursion (the checks use a 20,000-node chain), and check that 1,000 random trees survive the round trip. A 1,000-node tree takes 4 + 251 + 4,000 bytes this way, against about 10,000 characters of text.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def pack(root):
    ___

def unpack(data):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nT = TreeNode\ndef shape(n):\n    out, stack = [], [n]\n    while stack:\n        x = stack.pop()\n        if x is None:\n            out.append(None)\n        else:\n            out.append(x.val)\n            stack += [x.right, x.left]\n    return out\ndef rand_tree(rng, n):\n    if n == 0:\n        return None\n    k = rng.randint(0, n - 1)\n    return T(rng.randint(-2**31, 2**31 - 1), rand_tree(rng, k), rand_tree(rng, n - 1 - k))\ndef test_format():\n    \"the exact bytes\"\n    assert pack(None) == bytes([0, 0, 0, 0, 0])\n    assert pack(T(1, T(2), None)) == bytes([0, 0, 0, 2, 0b11000000, 0, 0, 0, 1, 0, 0, 0, 2])\n    assert pack(T(-1)) == bytes([0, 0, 0, 1, 0b10000000, 255, 255, 255, 255])\n    assert unpack(bytes([0, 0, 0, 0, 0])) is None\ndef test_roundtrip():\n    \"round trips\"\n    rng = random.Random(18)\n    for _ in range(1000):\n        t = rand_tree(rng, rng.randint(0, 12))\n        assert shape(unpack(pack(t))) == shape(t)\n    chain = None\n    for v in range(20_000):\n        chain = T(v, None, chain)\n    data = pack(chain)\n    assert len(data) == 4 + (2 * 20_000 + 1 + 7) // 8 + 4 * 20_000 and shape(unpack(data)) == shape(chain)\ndef test_bad():\n    \"malformed input\"\n    good = pack(T(1, T(2), None))\n    bad = [b\"\", good[:3], good[:-1], good + b\"\\x00\", bytes([0, 0, 0, 2, 0b10000000]) + bytes(8),\n           bytes([0, 0, 0, 1, 0b10100000]) + bytes(4), bytes([0, 0, 0, 1, 0b10000001]) + bytes(4),\n           bytes([0, 0, 0, 0, 0b10000000]), bytes([255, 255, 255, 255])]\n    for data in bad:\n        try:\n            unpack(data)\n            assert False, data\n        except ValueError:\n            pass\n    try:\n        pack(T(2**31))\n        assert False\n    except ValueError:\n        pass")

## Warm-up

Two different trees, one pre-order list: for a general binary tree, values alone lose the shape, which is why last week's format needed `#` markers. But only one of these is a BST. For a BST, the values themselves say where everything goes: 1 is less than 2, so it **must** be on the left.

## BSTs need no markers

Given the pre-order values of a BST, the first is the root; the values after it that are **smaller** form the left subtree, and the rest the right subtree. Splitting by searching for the first larger value at every level is O(n²) for a chain. To do it in O(n), pass an **upper bound** down, as on Day 1: build a subtree from the next values for as long as they're below the bound.

Each value is read once, so it's O(n). The same job can be done with an explicit stack, a monotonic stack like Week 43's: the stack holds the path of nodes still waiting for a right child, and each new value pops every node smaller than it (the last one popped is its parent, on the right); otherwise it's the left child of the top. That version has no recursion limit, and the exercises ask for it.

Saving is just the pre-order values. It works because the tree is a BST; if the data has been tampered with or the "BST" wasn't one, the load produces a different tree, so check with Day 1's `is_bst` or by comparing pre-orders.

## Level-order lists

Many puzzle websites write binary trees as a **level-order list**: values level by level, left to right, with `None` (or `null` in JSON) for a missing child, and trailing `None`s dropped. Only children of real nodes get a slot, so a sparse tree doesn't fill a whole level:

```text
        1
       / \          [1, 2, 3, None, 4, None, 5]
      2   3         2 has no left child; 3 has no left child;
       \   \        4 and 5 have no children (trailing Nones dropped)
        4   5
```

Loading uses a queue (Week 43): take the next parent from the queue, and give it the next two items from the list as its left and right children, adding any real ones to the queue.

## Trees as JSON

For trees whose nodes have any number of children (Week 45's general trees: folders, organisation charts, menus), the natural format is **nested JSON** (Week 23 Day 3): each node becomes an object holding its data and a list of its children.

Nested JSON is easy to read, and every language can load it. Its weakness is **depth**: each level of nesting is a level of recursion, both in your `to_dict` and inside the `json` module itself, which raises `RecursionError` for very deeply nested input. For trees that can be thousands of levels deep (a long chain of replies in a discussion, say), use a **flat** format instead: a list of `{"id": ..., "parent": ..., ...}` records, which is also how databases store trees (Week 45 Day 1 built a tree from a parent map).

## Formats change: versions and round trips

A format you save today must still load next year, after you've changed the program. Two habits help:

- **Put a version in the data.** Start the saved form with something like `{"format": "atlas-tree", "version": 2, ...}`. The loader checks the version, converts older ones, and refuses versions it doesn't know rather than guessing.
- **Test with round trips.** For any tree `t`, `load(save(t))` must equal `t`. Checking that over many random trees (including empty trees, single nodes, chains and awkward values like negative numbers or strings with spaces) catches most serialisation bugs. This is an example of **property-based testing**: instead of hand-picking inputs and outputs, you state a property that must always hold and test it on lots of generated inputs (the `hypothesis` library automates it).

Wrapping both directions in one class with `dumps` and `loads` methods (the names the `json` and `pickle` modules use) keeps them together, so they can't drift apart:

## Loading data you didn't write

Saved data comes back from files people can edit, from networks, from other programs. Treat it as **untrusted** (Week 37's defensive programming): a loader should either return a correct tree or raise one clear error, never crash with an `IndexError` from deep inside, loop for ever, or quietly build something wrong. Check:

- **Types**: is each node the shape you expect (a list of three, an object with the right keys)? Is each value the right type?
- **Structure**: does the data actually describe a tree? In a flat format, an id can appear twice, a parent can be missing, or parents can form a cycle.
- **Size and depth**: set limits, so a file of a billion nodes or a million levels is refused before it uses all your memory or overflows the stack.
- **Meaning**: if it's meant to be a BST, is it one?

And never load untrusted data with `eval` or `pickle`: both can run arbitrary code. The `pickle` documentation itself warns that it is "not secure" and that you should "only unpickle data you trust". JSON, plain text and careful checks are the safe route.

(`bool` is a subclass of `int` in Python, so `isinstance(True, int)` is `True`; a careful check excludes it explicitly.)

## Practice

## Quiz

## Daily challenge

## Before you go

Pick a format you've seen in the wild (a playlist file, a saved game, a settings file) and list what a careful loader for it should check. Tomorrow is the Forge: a tree-backed index with range queries, saved and loaded safely.